# Car Price Prediction: training notebook
Run the cells from top to bottom. Keep `Cardetails.csv` in the same folder as this notebook (in Colab, upload it to the Files panel first).

In [ ]:
import pandas as pd
import numpy as np
import pickle as pk

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

## 1. Load data

In [ ]:
cars_data = pd.read_csv('Cardetails.csv')
print(cars_data.shape)
cars_data.head()

## 2. Clean data (drop torque, nulls, duplicates)

In [ ]:
cars_data.drop(columns=['torque'], inplace=True)

print("Null values per column:")
print(cars_data.isnull().sum())

cars_data.dropna(inplace=True)
cars_data.drop_duplicates(inplace=True)
cars_data.reset_index(drop=True, inplace=True)

print("Shape after cleaning:", cars_data.shape)

## 3. Convert text columns to numbers

In [ ]:
def get_brand_name(car_name):
    return car_name.split(' ')[0].strip()

def clean_data(value):
    value = str(value).split(' ')[0].strip()
    if value == '':
        value = 0
    return float(value)

cars_data['name'] = cars_data['name'].apply(get_brand_name)

for col in ['mileage', 'engine', 'max_power']:
    cars_data[col] = cars_data[col].apply(clean_data)

cars_data[['name', 'mileage', 'engine', 'max_power']].head()

## 4. Encode categorical columns (same numbers as main.py)

In [ ]:
brands = ['Maruti', 'Skoda', 'Honda', 'Hyundai', 'Toyota', 'Ford', 'Renault',
          'Mahindra', 'Tata', 'Chevrolet', 'Datsun', 'Jeep', 'Mercedes-Benz',
          'Mitsubishi', 'Audi', 'Volkswagen', 'BMW', 'Nissan', 'Lexus',
          'Jaguar', 'Land', 'MG', 'Volvo', 'Daewoo', 'Kia', 'Fiat', 'Force',
          'Ambassador', 'Ashok', 'Isuzu', 'Opel']
brand_map = {brand: i + 1 for i, brand in enumerate(brands)}
owner_map = {'First Owner': 1, 'Second Owner': 2, 'Third Owner': 3,
             'Fourth & Above Owner': 4, 'Test Drive Car': 5}
fuel_map = {'Diesel': 1, 'Petrol': 2, 'LPG': 3, 'CNG': 4}
seller_map = {'Individual': 1, 'Dealer': 2, 'Trustmark Dealer': 3}
transmission_map = {'Manual': 1, 'Automatic': 2}

cars_data['name'] = cars_data['name'].map(brand_map)
cars_data['fuel'] = cars_data['fuel'].map(fuel_map)
cars_data['seller_type'] = cars_data['seller_type'].map(seller_map)
cars_data['transmission'] = cars_data['transmission'].map(transmission_map)
cars_data['owner'] = cars_data['owner'].map(owner_map)

# If any value was not in a map it becomes NaN; show where, then stop
missing = cars_data.isnull().sum()
print(missing[missing > 0])
assert cars_data.isnull().sum().sum() == 0, "A category was not mapped: check the unique values of that column"

cars_data.info()

## 5. Train / test split

In [ ]:
feature_cols = ['name', 'year', 'km_driven', 'fuel', 'seller_type', 'transmission',
                'owner', 'mileage', 'engine', 'max_power', 'seats']

input_data = cars_data[feature_cols]
output_data = cars_data['selling_price']

x_train, x_test, y_train, y_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42
)
print("Train:", x_train.shape, " Test:", x_test.shape)

## 6. Train and compare models

In [ ]:
# Linear Regression (baseline)
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)
lr_pred = lr_model.predict(x_test)
lr_r2 = r2_score(y_test, lr_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)

# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)
rf_pred = rf_model.predict(x_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)

print("Linear Regression -> R2:", round(lr_r2, 4), " MAE:", round(lr_mae, 2))
print("Random Forest     -> R2:", round(rf_r2, 4), " MAE:", round(rf_mae, 2))

## 7. Save the better model

In [ ]:
best_model = rf_model if rf_r2 > lr_r2 else lr_model
print("Saving:", type(best_model).__name__)

with open('model.pkl', 'wb') as f:
    pk.dump(best_model, f)

# Quick check with one sample car
sample = pd.DataFrame(
    [[5, 2022, 12000, 1, 1, 1, 1, 12.99, 2494.0, 100.6, 5.0]],
    columns=feature_cols
)
print("Sample prediction:", best_model.predict(sample)[0])

## 8. (Colab only) download model.pkl

In [ ]:
try:
    from google.colab import files
    files.download('model.pkl')
except ImportError:
    print("Not running in Colab: model.pkl is saved in the current folder.")